In [ ]:
from pathlib import Path
import json
import time
import gc

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss
)

PROJECT_DIR = Path("C:/Case Study Trees/HiggsStudy")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "artifacts" / "models"
INPUT_INFO_DIR = PROJECT_DIR / "artifacts" / "input_info"
OPTUNA_DIR = PROJECT_DIR / "optuna"
VALIDATION_DIR = PROJECT_DIR / "results" / "validation"

RANDOM_STATE = 42
N_THREADS = 4
TUNING_SECONDS = 30 * 60

with open(INPUT_INFO_DIR / "higgs_input_info.json") as file:
    input_info = json.load(file)

X_train = np.load(
    PROCESSED_DIR / "X_train.npy",
    mmap_mode="r"
)

y_train = np.load(
    PROCESSED_DIR / "y_train.npy",
    mmap_mode="r"
)

X_valid = np.load(
    PROCESSED_DIR / "X_valid.npy",
    mmap_mode="r"
)

y_valid = np.load(
    PROCESSED_DIR / "y_valid.npy",
    mmap_mode="r"
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.shape[1] == X_valid.shape[1]

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("Predictor dtype:", X_train.dtype)

print(
    "Training predictor array size:",
    round(X_train.nbytes / 1024**3, 2),
    "GiB"
)

print("Optuna budget per model:", TUNING_SECONDS, "seconds")
print("CPU threads per model:", N_THREADS)

c:\Users\gdeep\anaconda3\envs\100DaysML_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training shape: (9500000, 28)
Validation shape: (500000, 28)
Predictor dtype: float32
Training predictor array size: 0.99 GiB
Optuna budget per model: 1800 seconds
CPU threads per model: 4


## XGBOOST

In [9]:
from xgboost import XGBClassifier

xgb_baseline = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=N_THREADS,
    verbosity = 2
)

print("Training XGBoost...", flush=True)

start = time.perf_counter()
xgb_baseline.fit(X_train, y_train)
xgb_fit_seconds = time.perf_counter() - start

start = time.perf_counter()
xgb_valid_probability = xgb_baseline.predict_proba(X_valid)[:, 1]
xgb_prediction_seconds = time.perf_counter() - start

xgb_results = {
    "Model": "XGBoost",
    "Version": "Baseline",
    "Validation_Log_Loss": log_loss(
        y_valid, xgb_valid_probability
    ),
    "Validation_ROC_AUC": roc_auc_score(
        y_valid, xgb_valid_probability
    ),
    "Validation_AP": average_precision_score(
        y_valid, xgb_valid_probability
    ),
    "Fit_Seconds": xgb_fit_seconds,
    "Validation_Prediction_Seconds": xgb_prediction_seconds,
    "CPU_Threads": N_THREADS
}

joblib.dump(
    xgb_baseline,
    MODEL_DIR / "higgs_xgb_baseline.joblib"
)

xgb_baseline.save_model(
    str(MODEL_DIR / "higgs_xgb_baseline.json")
)

pd.DataFrame([xgb_results]).to_csv(
    VALIDATION_DIR / "xgb_baseline_validation.csv",
    index=False
)

display(pd.DataFrame([xgb_results]).round(4))

del xgb_baseline, xgb_valid_probability
gc.collect()

Training XGBoost...
[16:38:39] INFO: C:\actions-runner\_work\xgboost\xgboost\src\data\iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (9500000, 28, 266000000).


,Model,Version,Validation_Log_Loss,Validation_ROC_AUC,Validation_AP,Fit_Seconds,Validation_Prediction_Seconds,CPU_Threads
0,XGBoost,Baseline,0.5267,0.8121,0.8279,96.2474,0.3781,4


4869

## LIGHT GBM

In [5]:
from lightgbm import LGBMClassifier

lgb_baseline = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    num_leaves=31,
    objective="binary",
    random_state=RANDOM_STATE,
    n_jobs=N_THREADS,
    verbosity=2
)

print("Training LightGBM...", flush=True)

start = time.perf_counter()
lgb_baseline.fit(X_train, y_train)
lgb_fit_seconds = time.perf_counter() - start

start = time.perf_counter()
lgb_valid_probability = lgb_baseline.predict_proba(X_valid)[:, 1]
lgb_prediction_seconds = time.perf_counter() - start

lgb_results = {
    "Model": "LightGBM",
    "Version": "Baseline",
    "Validation_Log_Loss": log_loss(
        y_valid, lgb_valid_probability
    ),
    "Validation_ROC_AUC": roc_auc_score(
        y_valid, lgb_valid_probability
    ),
    "Validation_AP": average_precision_score(
        y_valid, lgb_valid_probability
    ),
    "Fit_Seconds": lgb_fit_seconds,
    "Validation_Prediction_Seconds": lgb_prediction_seconds,
    "CPU_Threads": N_THREADS
}

joblib.dump(
    lgb_baseline,
    MODEL_DIR / "higgs_lgb_baseline.joblib"
)

pd.DataFrame([lgb_results]).to_csv(
    VALIDATION_DIR / "lgb_baseline_validation.csv",
    index=False
)

display(pd.DataFrame([lgb_results]).round(4))

del lgb_baseline, lgb_valid_probability
gc.collect()

Training LightGBM...
[LightGBM] [Info] Number of positive: 5034018, number of negative: 4465982
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.078927
[LightGBM] [Debug] init for col-wise cost 0.000341 seconds, init for row-wise cost 0.882050 seconds
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.249414 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6132
[LightGBM] [Info] Number of data points in the train set: 9500000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.529897 -> initscore=0.119729
[LightGBM] [Info] Start training from score 0.119729
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 6
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 6
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 6
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 6
[LightGBM] [Debug] Trained a tree with leave

,Model,Version,Validation_Log_Loss,Validation_ROC_AUC,Validation_AP,Fit_Seconds,Validation_Prediction_Seconds,CPU_Threads
0,LightGBM,Baseline,0.5303,0.8091,0.825,92.5892,1.5331,4


4381

## CATBOOST

In [6]:
from catboost import CatBoostClassifier

cat_baseline = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    loss_function="Logloss",
    boosting_type="Plain",
    random_seed=RANDOM_STATE,
    thread_count=N_THREADS,
    allow_writing_files=False,
    verbose=10
)

print("Training CatBoost...", flush=True)

start = time.perf_counter()
cat_baseline.fit(X_train, y_train)
cat_fit_seconds = time.perf_counter() - start

start = time.perf_counter()
cat_valid_probability = cat_baseline.predict_proba(
    X_valid,
    thread_count=N_THREADS
)[:, 1]

cat_prediction_seconds = time.perf_counter() - start

cat_results = {
    "Model": "CatBoost",
    "Version": "Baseline",
    "Validation_Log_Loss": log_loss(
        y_valid, cat_valid_probability
    ),
    "Validation_ROC_AUC": roc_auc_score(
        y_valid, cat_valid_probability
    ),
    "Validation_AP": average_precision_score(
        y_valid, cat_valid_probability
    ),
    "Fit_Seconds": cat_fit_seconds,
    "Validation_Prediction_Seconds": cat_prediction_seconds,
    "CPU_Threads": N_THREADS
}

joblib.dump(
    cat_baseline,
    MODEL_DIR / "higgs_cat_baseline.joblib"
)

cat_baseline.save_model(
    str(MODEL_DIR / "higgs_cat_baseline.cbm")
)

pd.DataFrame([cat_results]).to_csv(
    VALIDATION_DIR / "cat_baseline_validation.csv",
    index=False
)

display(pd.DataFrame([cat_results]).round(4))

del cat_baseline, cat_valid_probability
gc.collect()

Training CatBoost...
0:	learn: 0.6754399	total: 2.23s	remaining: 3m 41s
10:	learn: 0.6071770	total: 20.3s	remaining: 2m 43s
20:	learn: 0.5847625	total: 36.8s	remaining: 2m 18s
30:	learn: 0.5722845	total: 53.9s	remaining: 2m
40:	learn: 0.5641150	total: 1m 10s	remaining: 1m 40s
50:	learn: 0.5585179	total: 1m 25s	remaining: 1m 22s
60:	learn: 0.5537764	total: 1m 43s	remaining: 1m 5s
70:	learn: 0.5499165	total: 2m	remaining: 49.3s
80:	learn: 0.5473832	total: 2m 18s	remaining: 32.5s
90:	learn: 0.5446346	total: 2m 38s	remaining: 15.7s
99:	learn: 0.5425873	total: 2m 53s	remaining: 0us


,Model,Version,Validation_Log_Loss,Validation_ROC_AUC,Validation_AP,Fit_Seconds,Validation_Prediction_Seconds,CPU_Threads
0,CatBoost,Baseline,0.5427,0.7999,0.8167,217.0351,0.2079,4


0

In [10]:
validation_results = pd.concat(
    [
        pd.read_csv(
            VALIDATION_DIR / f"{key}_baseline_validation.csv"
        )
        for key in ["xgb", "lgb", "cat"]
    ],
    ignore_index=True
)

validation_results = validation_results.sort_values(
    "Validation_Log_Loss"
).reset_index(drop=True)

validation_results.to_csv(
    VALIDATION_DIR / "all_baseline_validation_comparison.csv",
    index=False
)

display(validation_results.round(4))

,Model,Version,Validation_Log_Loss,Validation_ROC_AUC,Validation_AP,Fit_Seconds,Validation_Prediction_Seconds,CPU_Threads
0,XGBoost,Baseline,0.5267,0.8121,0.8279,96.2474,0.3781,4
1,LightGBM,Baseline,0.5303,0.8091,0.8250,92.5892,1.5331,4
2,CatBoost,Baseline,0.5427,0.7999,0.8167,217.0351,0.2079,4
